# PM2.5 Prediction Model

**Input:** `df_model_monthly.csv` built in `features.ipynb`.

| Model | Algorithm | Features                                              | Goal |
|---|---|-------------------------------------------------------|---|
| **A** | RF + XGBoost | environmental, spatial, contextual variables          | Policy story: what can municipalities act on? |
| **B** | RF | PM2.5 lags, rolling features, lagged pollutants, weather, context | Accuracy benchmark |
| **C** | Ridge | Same as A                                             | Linear baseline with signed coefficients |

**Two splits:** time (train past / test future) and spatial (train on some stations / test on unseen stations).

**Primary outputs:** files in `model_output/` consumed by `policy_translation.ipynb`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.base import clone
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
print("Libraries loaded")


## Load final modeling dataset

This CSV was created in `features.ipynb` and already contains:
- date
- station identifier (`eoi_code`)
- weather/context variables
- lag features
- rolling features

For now, we only use the columns needed for Model A.

In [ ]:
df = pd.read_csv("df_model_monthly.csv", parse_dates=["date"])

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Stations:", df["eoi_code"].nunique())

display(df.head())

In [ ]:
# Ensuring target exists, date is datetime, station id exists, PM2.5 has non-missing values
print("Target missing:", df["PM2_5"].isna().sum())
print("Date dtype:", df["date"].dtype)
print("Station id missing:", df["eoi_code"].isna().sum())

display(df[["date", "eoi_code", "PM2_5"]].head())

## Modeling dataset and splits

Before building individual models, we create one shared modeling dataset and one shared split setup.

This keeps Models A, B, and C fully comparable:
- same rows
- same target
- same station grouping
- same time split
- same spatial split

Filtering only on PM2.5 history features required by the forecasting setup.
We do not filter on pollutant lags like O3, because those reflect real sensor availability and would remove too many observations.

In [ ]:
required_history = [
    "PM2_5_lag1",
    "PM2_5_lag2",
    "PM2_5_lag3",
    "PM2_5_roll3_mean",
    "PM2_5_roll3_std"
]

df_model = df.dropna(subset=required_history).copy()

print("Rows before filtering:", len(df))
print("Rows after common history filter:", len(df_model))
print("Stations after filter:", df_model["eoi_code"].nunique())

display(df_model.head())

## Target and grouping variable

These objects will be reused by all models:
- `y` = PM2.5 target
- `groups` = station identifier for spatial split

In [ ]:
target_col = "PM2_5"
y = df_model[target_col].copy()
groups = df_model["eoi_code"].copy()

print("Filtered modeling shape:", df_model.shape)
print("Target shape:", y.shape)
print("Unique stations:", groups.nunique())

## Common train/test splits

We define the split and reuse it for all models.

### Time split
Train on earlier months, test on later months

In [ ]:
# Time split
cutoff_date = df_model["date"].quantile(0.8)
time_train_mask = df_model["date"] <= cutoff_date
time_test_mask = df_model["date"] > cutoff_date

print("Cutoff:", cutoff_date.date())
print("Train/test:", time_train_mask.sum(), time_test_mask.sum())

### Spatial split
Train on some stations, test on unseen stations

In [ ]:
# Spatial split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(df_model, y, groups=groups))

print("Spatial train/test rows:", len(tr_idx), len(te_idx))
print("Train stations:", df_model.iloc[tr_idx]["eoi_code"].nunique())
print("Test stations:", df_model.iloc[te_idx]["eoi_code"].nunique())

## Model A — RF + XGBoost (Explanatory)

### ⚠️ TO BUILD

Train RF and XGBoost on `features_a`. Pick champion by lowest average RMSE across both splits.

**Data ready to use:**
- `X_a` — built below
- `y`, `groups`, `time_train_mask`, `time_test_mask`, `tr_idx`, `te_idx` — shared from setup above
- `pre_a` — preprocessor built below

**XGBoost params:** `n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0`

**RF params:** `n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1`

**Variable names needed by comparison cell:**
- `pred_champ_a_t`, `pred_champ_a_s` — champion predictions on time/spatial test
- `m_champ_a_t`, `m_champ_a_s` — metrics dicts
- `champion_a_name` — string: `"RF"` or `"XGBoost"`
- `champ_a_full` — model retrained on full dataset (for SHAP / feature importance)

In [ ]:
# Model A feature set
# Keep this list simple and explicit.
# If a column name does not exist in df_model, it will be skipped safely.

candidate_features_a = [
    # Seasonality
    "Season",
    "month_sin",
    "month_cos",

    # Weather
    "Temp_Mean",
    "Wind_Speed",
    "Precipitation",

    # Lagged weather (still okay for explanatory model)
    "Temp_Mean_lag1",
    "Wind_Speed_lag1",
    "Precipitation_lag1",

    # Spatial
    "Altitude",
    "Latitude",
    "Longitude",
    "Green_Ratio",
    "Population_Density",

    # Contextual / station metadata
    "Station_Type",
    "Station_Area",
    "City"
]

features_a = [c for c in candidate_features_a if c in df_model.columns]

print("Model A features:", len(features_a))
print(features_a)


In [ ]:
def make_preprocessor(feature_list):
    num_cols = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols = [c for c in feature_list if c not in num_cols]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])

def reg_metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2":   r2_score(y_true, y_pred),
    }

pre_a = make_preprocessor(features_a)
print("Preprocessors and metrics helper ready")

### Build Model A feature matrix

We now create the feature matrix for Model A only.

Important:
- `y` is already shared
- split masks and indices are already shared
- only `X_a` is model-specific

In [ ]:
X_a = df_model[features_a].copy()

print("Model A X shape:", X_a.shape)
display(X_a.head())

# y, groups, time_train_mask, time_test_mask, tr_idx, te_idx
# were already created in the setup section above.

### Model A — Time split

The dataset is divided based on the date:
- the model is trained on **earlier months**
- the model is tested on **later months**

This simulates a realistic forecasting scenario, where we use past data to predict future air pollution levels.

We train both:
- Random Forest (RF)
- XGBoost (XGB)

and compare their performance using MAE, RMSE, and R².

In [ ]:
# Time split data
X_train_a_t = X_a.loc[time_train_mask] # Features for training(past data)
X_test_a_t = X_a.loc[time_test_mask] # Features for testing (future data)
y_train_t = y.loc[time_train_mask] # values for training
y_test_t = y.loc[time_test_mask] # values for testing

# RF - time split
# Build Pipeline: Preprocessig + RF

rf_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])
# Train model on past data
rf_a_t.fit(X_train_a_t, y_train_t)
pred_rf_a_t = rf_a_t.predict(X_test_a_t) #Predict PM2.5 on future data
m_rf_a_t = reg_metrics(y_test_t, pred_rf_a_t) #Evaluate predictions using MAE, RMSE, and R²
print("RF time split:", m_rf_a_t)

# XGBoost — time split
# Same process as RF
xgb_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_t.fit(X_train_a_t, y_train_t)
pred_xgb_a_t = xgb_a_t.predict(X_test_a_t)
m_xgb_a_t = reg_metrics(y_test_t, pred_xgb_a_t)

print("XGB time split:", m_xgb_a_t)

### Model A — Spatial split

Instead of splitting the data by time, we split it by **monitoring stations** (`eoi_code`).

- the model is trained on data from a subset of stations
- the model is tested on **completely unseen stations**

We train both:
- Random Forest(RF)
- XGBoost(XGB)

In [ ]:
# Spatial split data
# Split the dataset using indices created earlier (GroupShuffleSplit)
# Ensures that train and test contain DIFFERENT stations

X_train_a_s = X_a.iloc[tr_idx] # Features for training (some stations)
X_test_a_s = X_a.iloc[te_idx] # Features for testing (unseen stations)
y_train_s = y.iloc[tr_idx] #
y_test_s = y.iloc[te_idx]

# RF — spatial split
rf_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])
rf_a_s.fit(X_train_a_s, y_train_s)
pred_rf_a_s = rf_a_s.predict(X_test_a_s)
m_rf_a_s = reg_metrics(y_test_s, pred_rf_a_s)
print("RF spatial split:", m_rf_a_s)

# XGBoost — spatial split
xgb_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_s.fit(X_train_a_s, y_train_s)
pred_xgb_a_s = xgb_a_s.predict(X_test_a_s)
m_xgb_a_s = reg_metrics(y_test_s, pred_xgb_a_s)
print("XGB spatial split:", m_xgb_a_s)


### Model A — Model Selection

Pick the model with the lowest average RMSE across both splits.

In [ ]:
# Champion selection by lowest avg RMSE across both splits
avg_rmse_rf = (m_rf_a_t["RMSE"] + m_rf_a_s["RMSE"]) / 2
avg_rmse_xgb = (m_xgb_a_t["RMSE"] + m_xgb_a_s["RMSE"]) / 2

print(f"RF avg RMSE: {avg_rmse_rf:.4f}")
print(f"XGB avg RMSE: {avg_rmse_xgb:.4f}")

if avg_rmse_rf <= avg_rmse_xgb:
    champion_a_name = "RF"
    pred_champ_a_t = pred_rf_a_t
    pred_champ_a_s = pred_rf_a_s
    m_champ_a_t = m_rf_a_t
    m_champ_a_s = m_rf_a_s
else:
    champion_a_name = "XGBoost"
    pred_champ_a_t = pred_xgb_a_t
    pred_champ_a_s = pred_xgb_a_s
    m_champ_a_t = m_xgb_a_t
    m_champ_a_s = m_xgb_a_s

print(f"\nChampion A: {champion_a_name}")
print(f"Time split   - RMSE={m_champ_a_t['RMSE']:.4f}  MAE={m_champ_a_t['MAE']:.4f}  R2={m_champ_a_t['R2']:.4f}")
print(f"Spatial split - RMSE={m_champ_a_s['RMSE']:.4f}  MAE={m_champ_a_s['MAE']:.4f}  R2={m_champ_a_s['R2']:.4f}")

# Retrain champion on FULL dataset (for SHAP / feature importance)
if champion_a_name == "RF":
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )),
    ])
else:
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )),
    ])

champ_a_full.fit(X_a, y)
print(f"\nchamp_a_full retrained on full dataset ({len(X_a)} rows).")


### Model A — Summary table

In [ ]:
metrics_a = pd.DataFrame([
    {"split": "time", "model": f"RF_A", **m_rf_a_t},
    {"split": "spatial", "model": f"RF_A", **m_rf_a_s},
    {"split": "time", "model": f"XGBoost_A", **m_xgb_a_t},
    {"split": "spatial", "model": f"XGBoost_A", **m_xgb_a_s},
])
display(metrics_a)
metrics_a.to_csv("model_output/model_a_metrics.csv", index=False)
print(f"Champion: {champion_a_name}")


### Model A — Performance Comparison (RF vs XGBoost)

In [ ]:
plot_a = metrics_a.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_a,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model A — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modela_matrix_comparison.png", scale=2)
fig.show()


### Model A — Residuals

In [ ]:
# residuals
r_rf_a_s = y_test_s.values - pred_rf_a_s
r_xgb_a_s = y_test_s.values - pred_xgb_a_s

# common bins so both plots are comparable
all_r_s = np.concatenate([r_rf_a_s, r_xgb_a_s])
bins_s = np.histogram_bin_edges(all_r_s, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (spatial test)", "Model A XGBoost residuals (spatial test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (spatial test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_space_rf_xgb.png", scale=2)
fig.show()

In [ ]:
# residuals
r_rf_a_t = y_test_t.values - pred_rf_a_t
r_xgb_a_t = y_test_t.values - pred_xgb_a_t

# common bins so both plots are comparable
all_r_t = np.concatenate([r_rf_a_t, r_xgb_a_t])
bins_t = np.histogram_bin_edges(all_r_t, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (time test)", "Model A XGBoost residuals (time test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (time test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_time_rf_xgb.png", scale=2)
fig.show()

### Model A — SHAP (RF)

In [ ]:
import scipy.sparse as sp
import shap

# Refit Random Forest on the full Model A dataset
# we want interpretation of the final Model A RF, not split-based evaluation
rf_a_full = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )),
])
rf_a_full.fit(X_a, y)

# same preprocessing used inside the pipeline
Xp = rf_a_full.named_steps["preprocess"].transform(X_a)


if sp.issparse(Xp):
    Xp = Xp.toarray()

Xp = np.asarray(Xp, dtype=np.float64)

# sample so SHAP runs faster
rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

# compute SHAP values for sample
explainer = shap.TreeExplainer(rf_a_full.named_steps["model"])
sv = explainer.shap_values(Xs)

# names of features after preprocessing
fn = rf_a_full.named_steps["preprocess"].get_feature_names_out()

# global SHAP importance = average absolute SHAP value per feature
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_rf_a = (
    pd.DataFrame({
        "feature": fn,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_rf_a.to_csv("model_output/modela_rf_shap_importance.csv", index=False)
display(shap_importance_rf_a.head(15))

In [ ]:
# Beeswarm-style Plotly view for top features
top_n = 15
top_features = shap_importance_rf_a.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})

long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

# normalize feature values
long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm_rf_a = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    range_color=[-2.5, 2.5],
    opacity=0.75,
    title=f"SHAP — Model A (RF): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm_rf_a.update_traces(marker={"size": 6})
fig_swarm_rf_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm_rf_a.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm_rf_a.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm_rf_a.write_image("images/modela_rf_shap_beeswarm.png", scale=2)
fig_swarm_rf_a.show()

### Model A — SHAP (XGBOOST)

In [ ]:
# Refit XGBOOST on the full dataset for final interpretation
xgb_a_full = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )),
])

xgb_a_full.fit(X_a, y)

# preprocessing
Xp = xgb_a_full.named_steps["preprocess"].transform(X_a)

if sp.issparse(Xp):
    Xp = Xp.toarray()

Xp = np.asarray(Xp, dtype=np.float64)

# sample rows for SHAP
rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

# tree SHAP for XGBoost
explainer = shap.TreeExplainer(xgb_a_full.named_steps["model"])
sv = explainer.shap_values(Xs)

fn = xgb_a_full.named_steps["preprocess"].get_feature_names_out()

# global importance table
mean_abs_shap = np.abs(sv).mean(axis=0)

shap_importance_xgb_a = (
    pd.DataFrame({
        "feature": fn,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_xgb_a.to_csv("model_output/modela_xgb_shap_importance.csv", index=False)

display(shap_importance_xgb_a.head(15))

In [ ]:
top_n = 15
top_features = shap_importance_xgb_a.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})

long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

# Normalize values for color mapping
long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

# Add jitter so points spread vertically
jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm_xgb_a = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    range_color=[-2.5, 2.5],
    opacity=0.75,
    title=f"SHAP — Model A (XGBoost): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm_xgb_a.update_traces(marker={"size": 6})

fig_swarm_xgb_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm_xgb_a.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm_xgb_a.add_vline(x=0, line_dash="dash", line_color="black")

fig_swarm_xgb_a.write_image("images/modela_xgb_shap_beeswarm.png", scale=2)
fig_swarm_xgb_a.show()

## Model B — Random Forest (Environmental + PM2.5 History + Lagged Pollutants)

Accuracy benchmark. Adds PM2.5 lags, rolling stats, and lagged co-pollutants on top of all Model A features.
Comparing A vs B shows how much pollution memory and co-pollutant history improves forecasting.

In [ ]:
# Model B feature set
features_b = [
    "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean", "PM2_5_roll3_std",
    "PM10_lag1", "NO2_lag1", "O3_lag1",
    "Temp_Mean", "Wind_Speed", "Precipitation",
    "Temp_Mean_lag1", "Wind_Speed_lag1", "Precipitation_lag1",
    "month_sin", "month_cos",
    "Altitude", "Latitude", "Longitude",
    "Green_Ratio", "Population_Density",
    "Station_Type", "Station_Area",
]
features_b = [c for c in features_b if c in df_model.columns]
X_b = df_model[features_b].copy()

print("Model B features:", len(features_b))
print(features_b)
print("Model B X shape:", X_b.shape)

### Model B — Time split

In [ ]:
def make_preprocessor_b(feature_list):
    num_cols_b = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols_b = [c for c in feature_list if c not in num_cols_b]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols_b),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols_b),
    ])

pre_b = make_preprocessor_b(features_b)

# Time split
X_train_b_t = X_b.loc[time_train_mask]
X_test_b_t = X_b.loc[time_test_mask]
y_train_t = y.loc[time_train_mask]
y_test_t = y.loc[time_test_mask]

rf_b = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b.fit(X_train_b_t, y_train_t)
pred_rf_b_t = rf_b.predict(X_test_b_t)
m_rf_b_t = reg_metrics(y_test_t, pred_rf_b_t)
print("Model B time split:", m_rf_b_t)

### Model B — Spatial split

In [ ]:
X_train_b_s = X_b.iloc[tr_idx]
X_test_b_s = X_b.iloc[te_idx]
y_train_s = y.iloc[tr_idx]
y_test_s = y.iloc[te_idx]

rf_b_s = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_s.fit(X_train_b_s, y_train_s)
pred_rf_b_s = rf_b_s.predict(X_test_b_s)
m_rf_b_s = reg_metrics(y_test_s, pred_rf_b_s)
print("Model B spatial split:", m_rf_b_s)

In [ ]:
metrics_b = pd.DataFrame([
    {"split": "time",    "model": "RandomForest_B", **m_rf_b_t},
    {"split": "spatial", "model": "RandomForest_B", **m_rf_b_s},
])
display(metrics_b)
metrics_b.to_csv("model_output/modelb_results.csv", index=False)
print("Saved to model_output/modelb_results.csv")

In [ ]:
plot_b = metrics_b.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_b,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model B — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modelb_metrics.png", scale=2)
fig.show()


In [ ]:
r = y_test_t.values - pred_rf_b_t
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (time test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_time.png", scale=2)
fig.show()


In [ ]:
r = y_test_s.values - pred_rf_b_s
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (spatial test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_space.png", scale=2)
fig.show()


### SHAP test 

SHAP helps us explain **why** Model B predicts higher or lower PM2.5 for each data point.

How it works:
- For each row, SHAP assigns a contribution value to every feature.
- A **positive SHAP value** means that feature pushes the prediction **up**.
- A **negative SHAP value** means that feature pushes the prediction **down**.
- Bigger absolute values mean stronger influence.

Why we need it:
- Random Forest is accurate but not directly interpretable.
- SHAP gives transparent feature-level evidence for the policy story.
- We use it to compare with Ridge signs (Model C) and check consistency across models.



In [ ]:
import scipy.sparse as sp
import shap

rf_b_full = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_full.fit(X_b, y)

Xp = rf_b_full.named_steps["preprocess"].transform(X_b)
if sp.issparse(Xp):
    Xp = Xp.toarray()
Xp = np.asarray(Xp, dtype=np.float64)

rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

explainer = shap.TreeExplainer(rf_b_full.named_steps["model"])
sv = explainer.shap_values(Xs)
fn = rf_b_full.named_steps["preprocess"].get_feature_names_out()

# Global SHAP importance (numeric)
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_df = (
    pd.DataFrame({"feature": fn, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
)

# 1) Per-point beeswarm direction view in Plotly
top_n = 15
top_features = shap_importance_df.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})
long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

# Per-feature normalization (z-score), then clip for stable color contrast
long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

# Beeswarm jitter
jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    # strict blue -> pink, no white midpoint
    color_continuous_scale=[
        [0.0, "#1f6bff"],   # low values
        [1.0, "#ff4fa3"],   # high values
    ],
    range_color=[-2.5, 2.5],  # enforce consistent, vivid mapping
    opacity=0.75,
    title=f"SHAP — Model B (RF): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm.update_traces(marker={"size": 6})
fig_swarm.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm.write_image("images/modelb_shap_beeswarm.png", scale=2)
fig_swarm.show()




**How the color scale is formed (SHAP beeswarm)**

Colors show feature value level, not SHAP magnitude.
For each feature, values are standardized with:

[ z = \frac{x - \mu}{\sigma + 10^{-9}} ]

So:

- Blue = lower-than-average value (negative (z))
- Pink = higher-than-average value (positive (z))

This makes colors comparable across features with different units.

In [ ]:
# 2) Top 15 features in Plotly
top15 = shap_importance_df.head(15).copy().iloc[::-1]
fig_top = px.bar(
    top15,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="SHAP — Model B (RF): top 15 mean |SHAP|",
)
fig_top.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig_top.write_image("images/modelb_shap_top15.png", scale=2)
fig_top.show()


In [ ]:
# 3) Summary table + export
top10_shap = shap_importance_df.head(10).copy()
print("Top 10 SHAP features (mean |SHAP|):")
display(top10_shap)
shap_importance_df.to_csv("model_output/modelb_shap_importance.csv", index=False)
print("Saved to model_output/modelb_shap_importance.csv")

## Model C — Ridge Regression (Linear Baseline)

Train Ridge on the same Model B feature matrix, evaluate on both time and spatial splits, and report signed coefficients for interpretation.

How to read this section:
- Higher R2 and lower RMSE/MAE indicate better fit.
- Coefficient sign gives direction (positive increases predicted PM2.5, negative decreases it).
- Coefficients should be interpreted alongside SHAP because one-hot encoding and correlated variables can spread linear weights.




In [ ]:
# 1. Feature Selection: model A
features_c = features_a.copy()
X_c = df_model[features_c].copy()

print(f"Model C features: {len(features_c)}")
print(f"Model C X shape: {X_c.shape}")


### Training of model C
We train a Ridge Regression model using the same preprocessing pipeline as Model A.

The C model is evaluated on two splits:
- **Time split**: tests how well the model predicts future observations.
- **Spatial split**: tests how well the model generalizes across different locations.

Ridge is used as a linear baseline model, allowing us to compare its performance with more complex models.

In [ ]:
# 2. Training
ridge_pipe = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])

# Time Split
X_train_c_t = X_c.loc[time_train_mask]
X_test_c_t  = X_c.loc[time_test_mask]
y_train_t   = y.loc[time_train_mask]
y_test_t    = y.loc[time_test_mask]

ridge_pipe.fit(X_train_c_t, y_train_t)
pred_ridge_t = ridge_pipe.predict(X_test_c_t)
m_ridge_t    = reg_metrics(y_test_t, pred_ridge_t)
print("Ridge time split:   ", m_ridge_t)

# Spatial Split
X_train_c_s = X_c.iloc[tr_idx]
X_test_c_s  = X_c.iloc[te_idx]
y_train_s   = y.iloc[tr_idx]
y_test_s    = y.iloc[te_idx]

ridge_pipe_s = clone(ridge_pipe)
ridge_pipe_s.fit(X_train_c_s, y_train_s)
pred_ridge_s = ridge_pipe_s.predict(X_test_c_s)
m_ridge_s    = reg_metrics(y_test_s, pred_ridge_s)
print("Ridge spatial split:", m_ridge_s)

# Metrics Table & Export
metrics_c = pd.DataFrame([
    {"split": "time",    "model": "Ridge_C", **m_ridge_t},
    {"split": "spatial", "model": "Ridge_C", **m_ridge_s},
])
metrics_c.to_csv("model_output/model_c_metrics.csv", index=False)
display(metrics_c)



#### Model C — Performance Evaluation

The Ridge model achieves:

- **Time split**
  - MAE ≈ 5.21
  - RMSE ≈ 10.00
  - R² ≈ 0.35

- **Spatial split**
  - MAE ≈ 4.92
  - RMSE ≈ 6.84
  - R² ≈ 0.32

The model shows moderate predictive performance, capturing some variation in PM2.5 levels, but likely missing non-linear relationships present in the data.

Performance is slightly better in the spatial split, indicating reasonable generalization across locations.

### Model C — Coefficient Analysis

To interpret the Ridge model, we retrain it on the full dataset and extract the learned coefficients.

Each coefficient represents:
- **Positive value** → increases predicted PM2.5
- **Negative value** → decreases predicted PM2.5
- **Bigger coefficients** → stronger effect

Features are ranked by their impact to identify the most important drivers.

In [ ]:
# 3. Full Retrain and Coefficient Analysis
ridge_full = clone(ridge_pipe)
ridge_full.fit(X_c, y)

feat_names_c = ridge_full.named_steps["preprocess"].get_feature_names_out()
coef_values = ridge_full.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feat_names_c,
    "coef": coef_values
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values(by="abs_coef", ascending=False)

# Visualization
top20_coef = coef_df.head(20).sort_values("coef")
max_abs = top20_coef["coef"].abs().max()

fig_c = px.bar(
    top20_coef,
    x="coef",
    y="feature",
    orientation="h",
    color="coef",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    title="Model C (Ridge) — Top 20 Drivers of PM2.5",
    labels={"coef": "Coefficient Value", "feature": "Feature"}
)

fig_c.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700
)
fig_c.update_coloraxes(cmin=-max_abs, cmax=max_abs)
fig_c.add_vline(x=0, line_dash="dash", line_color="black")
fig_c.write_image("images/modelc_coefficients.png", scale=2)
fig_c.show()

**Top Drivers of PM2.5 (Ridge Model) Plot**

The chart shows the 20 most influential features in the Ridge model:
- Several **city indicators** have the strongest impact, suggesting location plays a major role.
- Some cities are associated with **higher pollution levels**, while others show lower predicted values.
- Temperature also appears among the important variables, indicating a relationship with PM2.5 levels.


In [ ]:
# Visualization
coef_df.to_csv("model_output/model_c_coefficients.csv", index=False)

print("Top 10 Positive Coefficients (Increases PM2.5)")
display(coef_df.sort_values("coef", ascending=False).head(10)[["feature", "coef"]])

print("Top 10 Negative Coefficients (Decreases PM2.5)")
display(coef_df.sort_values("coef", ascending=True).head(10)[["feature", "coef"]])

**Top Positive Coefficients (Increase PM2.5)**

These features are associated with higher predicted PM2.5 levels.

We observed that some cities (Ceglie Messapica, Alessandria) have strong positive coefficients.
Therefore, these locations tend to have higher pollution levels compared to others in the dataset.

These effects likely capture local environmental or urban characteristics.

**Top Negative Coefficients (Decrease PM2.5)**

These features are associated with lower predicted PM2.5 levels.
We observed that some cities (Teramo, Lecco) show strong negative coefficients.
Temperature has a negative coefficient, showing that higher temperatures may be linked to lower PM2.5 levels.


### Model B vs Model C — Feature Effect Comparison

This table compares:
- **Model B (RF + SHAP):** global importance (`mean_abs_shap`)
- **Model C (Ridge):** signed linear coefficient (`ridge_coef`)

How to read it:
- High `mean_abs_shap` means the feature matters a lot for Model B predictions.
- `ridge_coef` sign gives direction in Model C (positive increases PM2.5, negative decreases PM2.5).
- Features with high SHAP and stable Ridge sign are stronger candidates for reporting in the final story.

In [ ]:
shap_cmp = shap_importance_df.set_index("feature").copy()
ridge_cmp = coef_full.rename("ridge_coef").to_frame()

comparison_bc = (
    shap_cmp
    .join(ridge_cmp, how="inner")
    .assign(
        abs_ridge_coef=lambda d: d["ridge_coef"].abs(),
        ridge_direction=lambda d: np.where(d["ridge_coef"] >= 0, "increase", "decrease"),
    )
    .sort_values("mean_abs_shap", ascending=False)
)

comparison_bc["rank_shap"] = np.arange(1, len(comparison_bc) + 1)
comparison_bc["rank_abs_ridge"] = comparison_bc["abs_ridge_coef"].rank(ascending=False, method="dense").astype(int)

print("Top 20 transformed features: Model B SHAP vs Model C Ridge")
display(comparison_bc.head(20))

comparison_bc.to_csv("model_output/modelb_modelc_feature_comparison.csv")
print("Saved to model_output/modelb_modelc_feature_comparison.csv")

We prioritize rows with high mean_abs_shap and large abs_ridge_coef for stronger cross-model signals. 'ridge_direction' provides a simple directional summary from the linear baseline.

**Similarities**

Strong cross-model signals (high SHAP + high |Ridge|)

- num__PM2_5_lag1: SHAP = 3.589 (rank 1), Ridge = +1.735 (|coef| rank 5)
→ strongest global driver in RF and clearly positive in Ridge.
- num__month_sin: SHAP = 1.100 (rank 3), Ridge = -1.867 (|coef| rank 4)
→ strong seasonal effect with negative linear direction.
- num__month_cos: SHAP = 1.480 (rank 2), Ridge = +1.350 (|coef| rank 6)
→ also strong seasonality, positive linear direction.
- num__PM2_5_roll3_mean: SHAP = 0.430 (rank 9), Ridge = +1.340 (rank 7)
→ medium RF importance, strong positive linear memory effect.
- num__PM10_lag1: SHAP = 0.096 (rank 19), Ridge = +1.199 (rank 8)
→ weak in RF but relatively strong linear positive contribution.

Weak cross-model signals (high SHAP + high |Ridge|)
- num__Green_Ratio: SHAP = 0.050 (rank 20), Ridge = -0.072 (rank 26)
→ very weak in both models in current setup.
- num__Latitude: SHAP = 0.187 (rank 15), Ridge = +0.471 (rank 23)
→ modest/low effect.
- num__Precipitation: SHAP = 0.462 (rank 7), Ridge = +0.412 (rank 24)
→ moderate in RF, small linear coefficient.

**Differences**
- num__PM2_5_lag2: SHAP = 0.662 (rank 5) but Ridge = +3.574 (rank 1)
→ Ridge gives this very large weight; likely shared signal with other lag features.
- num__PM2_5_lag3: SHAP = 0.162 (rank 16) but Ridge = -2.200 (rank 2)
→ large negative coefficient despite low RF importance (classic collinearity pattern with lag1/lag2/rolling stats).
- num__PM2_5_roll3_std: SHAP = 0.198 (rank 14) but Ridge = +2.039 (rank 3)
→ linear model emphasizes volatility more than RF SHAP does.

We notice a very important problem here - collinearity.